<a href="https://colab.research.google.com/github/akshraj-git/hellyeah/blob/main/spacefight7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import HTML, display

space_shooter_html = """
<!DOCTYPE html>
<html>
<head>
<style>
    body {
        background-color: #050510;
        color: #ffffff;
        font-family: 'Consolas', 'Courier New', monospace;
        text-align: center;
        margin: 0;
        padding: 10px;
        user-select: none;
    }
    #gameCanvas {
        background: radial-gradient(circle at center, #0b0b26 0%, #020208 100%);
        border: 3px solid #00f0ff;
        border-radius: 10px;
        box-shadow: 0 0 25px rgba(0, 240, 255, 0.4);
        cursor: crosshair;
    }
    .hud {
        display: flex;
        justify-content: center;
        gap: 40px;
        font-size: 18px;
        font-weight: bold;
        margin-bottom: 8px;
    }
    .player-hp { color: #00f0ff; }
    .boss-hp { color: #ff0055; }
    .ai-status { color: #ffe600; }
    .controls {
        font-size: 13px;
        color: #888;
        margin-top: 6px;
    }
</style>
</head>
<body>

    <h2>🚀 SPACE SHOOTER vs ADAPTIVE AI BOSS 👾</h2>
    <div class="hud">
        <div class="player-hp">🛡️ PLAYER: <span id="playerHp">100</span> HP</div>
        <div class="ai-status">🧠 AI LEARNING: <span id="aiPattern">Analyzing...</span></div>
        <div class="boss-hp">👑 AI BOSS: <span id="bossHp">200</span> HP</div>
    </div>

    <canvas id="gameCanvas" width="700" height="500"></canvas>
    <div class="controls">Controls: <b>Move Mouse</b> to fly & aim | <b>Left Click / Space</b> to shoot</div>

<script>
const canvas = document.getElementById('gameCanvas');
const ctx = canvas.getContext('2d');

// --- GAME STATE ---
let player = { x: 350, y: 420, radius: 15, hp: 100, maxHp: 100 };
let boss = { x: 350, y: 80, radius: 30, hp: 200, maxHp: 200, vx: 3 };

let playerLasers = [];
let bossLasers = [];
let stars = [];

// Player Movement History (for AI Learning)
let playerHistory = [];
let aiPredictiveOffset = 0;
let lastShootTime = 0;
let gameOver = false;

// Generate Background Stars
for (let i = 0; i < 60; i++) {
    stars.push({ x: Math.random() * canvas.width, y: Math.random() * canvas.height, size: Math.random() * 2, speed: Math.random() * 1.5 + 0.5 });
}

// Mouse Controls
let mouse = { x: 350, y: 420, isDown: false };
canvas.addEventListener('mousemove', (e) => {
    const rect = canvas.getBoundingClientRect();
    mouse.x = e.clientX - rect.left;
    mouse.y = e.clientY - rect.top;
});
canvas.addEventListener('mousedown', () => mouse.isDown = true);
canvas.addEventListener('mouseup', () => mouse.isDown = false);

window.addEventListener('keydown', (e) => { if (e.code === 'Space') mouse.isDown = true; });
window.addEventListener('keyup', (e) => { if (e.code === 'Space') mouse.isDown = false; });

// --- AI PREDICTIVE MODEL ---
function updateAiModel() {
    // Record player's X position to calculate average movement velocity
    playerHistory.push(player.x);
    if (playerHistory.length > 20) playerHistory.shift();

    if (playerHistory.length > 5) {
        let velocity = (playerHistory[playerHistory.length - 1] - playerHistory[0]) / playerHistory.length;

        // AI Model predicts where player will be in 0.5 seconds
        aiPredictiveOffset = velocity * 15;

        let statusText = "Predicting Movement";
        if (Math.abs(velocity) < 0.5) statusText = "Target Stationary";
        else if (velocity > 0) statusText = "Predicting Right Strafe";
        else statusText = "Predicting Left Strafe";

        document.getElementById('aiPattern').innerText = statusText;
    }
}

// --- GAME LOOP ---
function gameLoop() {
    if (gameOver) return;

    ctx.clearRect(0, 0, canvas.width, canvas.height);

    // 1. Draw Moving Starfield Background
    ctx.fillStyle = '#ffffff';
    stars.forEach(s => {
        s.y += s.speed;
        if (s.y > canvas.height) s.y = 0;
        ctx.fillRect(s.x, s.y, s.size, s.size);
    });

    // 2. Update Player Position
    player.x += (mouse.x - player.x) * 0.15;
    player.y += (mouse.y - player.y) * 0.15;
    player.x = Math.max(20, Math.min(canvas.width - 20, player.x));
    player.y = Math.max(250, Math.min(canvas.height - 20, player.y));

    // Draw Player Ship (Blue Neon Triangle)
    ctx.shadowBlur = 15;
    ctx.shadowColor = '#00f0ff';
    ctx.fillStyle = '#00f0ff';
    ctx.beginPath();
    ctx.moveTo(player.x, player.y - 18);
    ctx.lineTo(player.x - 14, player.y + 14);
    ctx.lineTo(player.x + 14, player.y + 14);
    ctx.closePath();
    ctx.fill();

    // 3. Update Boss Position
    boss.x += boss.vx;
    if (boss.x < 60 || boss.x > canvas.width - 60) boss.vx *= -1;

    // Draw AI Boss (Red Glowing Hexagon)
    ctx.shadowColor = '#ff0055';
    ctx.fillStyle = '#ff0055';
    ctx.beginPath();
    for (let i = 0; i < 6; i++) {
        let angle = (i * Math.PI) / 3;
        let bx = boss.x + boss.radius * Math.cos(angle);
        let by = boss.y + boss.radius * Math.sin(angle);
        if (i === 0) ctx.moveTo(bx, by);
        else ctx.lineTo(bx, by);
    }
    ctx.closePath();
    ctx.fill();
    ctx.shadowBlur = 0;

    // 4. Player Shooting Logic
    let now = Date.now();
    if (mouse.isDown && now - lastShootTime > 120) {
        playerLasers.push({ x: player.x, y: player.y - 18, vy: -10 });
        lastShootTime = now;
    }

    // Update & Draw Player Lasers
    ctx.fillStyle = '#00ffff';
    playerLasers.forEach((l, i) => {
        l.y += l.vy;
        ctx.fillRect(l.x - 2, l.y, 4, 12);

        // Hit Boss Detection
        let dist = Math.hypot(l.x - boss.x, l.y - boss.y);
        if (dist < boss.radius) {
            boss.hp = Math.max(0, boss.hp - 2);
            playerLasers.splice(i, 1);
        }
    });
    playerLasers = playerLasers.filter(l => l.y > 0);

    // 5. AI Boss Adaptive Attack Logic
    updateAiModel();

    if (Math.random() < 0.06) {
        // AI shoots toward player's PREDICTED location
        let targetX = player.x + aiPredictiveOffset;
        let dx = targetX - boss.x;
        let dy = player.y - boss.y;
        let angle = Math.atan2(dy, dx);

        bossLasers.push({
            x: boss.x,
            y: boss.y + 20,
            vx: Math.cos(angle) * 6,
            vy: Math.sin(angle) * 6
        });
    }

    // Update & Draw AI Lasers
    ctx.fillStyle = '#ff0055';
    bossLasers.forEach((l, i) => {
        l.x += l.vx;
        l.y += l.vy;
        ctx.beginPath();
        ctx.arc(l.x, l.y, 5, 0, Math.PI * 2);
        ctx.fill();

        // Hit Player Detection
        let dist = Math.hypot(l.x - player.x, l.y - player.y);
        if (dist < player.radius + 5) {
            player.hp = Math.max(0, player.hp - 5);
            bossLasers.splice(i, 1);
        }
    });
    bossLasers = bossLasers.filter(l => l.y < canvas.height && l.x > 0 && l.x < canvas.width);

    // Update HUD Values
    document.getElementById('playerHp').innerText = player.hp;
    document.getElementById('bossHp').innerText = boss.hp;

    // Check Win/Loss Conditions
    if (player.hp <= 0 || boss.hp <= 0) {
        gameOver = true;
        ctx.fillStyle = 'rgba(0, 0, 0, 0.85)';
        ctx.fillRect(0, 0, canvas.width, canvas.height);
        ctx.fillStyle = '#ffffff';
        ctx.font = '28px Consolas';
        ctx.textAlign = 'center';

        if (boss.hp <= 0) {
            ctx.fillText('🎉 VICTORY! YOU DESTROYED THE AI BOSS! 🎉', canvas.width / 2, canvas.height / 2);
        } else {
            ctx.fillText('☠️ GAME OVER - THE AI MODEL ADAPTED & WON ☠️', canvas.width / 2, canvas.height / 2);
        }
        return;
    }

    requestAnimationFrame(gameLoop);
}

gameLoop();
</script>
</body>
</html>
"""

display(HTML(space_shooter_html))